To import necessary Python packages.

In [44]:
!pip3 install numpy pandas

In [45]:
import math
import numpy as np
import numpy.typing as npt
import pandas as pd
from typing import List
import gc
from collections import defaultdict
from dbm import error
from dataclasses import dataclass, field

## Table of Acronyms and Parameters
| Parameter | Description | Maths |
| :--- | :--- |:---|
| $P_c$ | Parallelism of the base-matrix columns ||
| $P_r$ | Size of a row chunk, which indicates the length of a barrel shifter under a L1BS component ||
| $W^s$ | The number of the implemented L1BS components for one parallelised base-matrix column, each of which performs the message-passing operation for a row chunk. Namely, the number of row chunks processed in parallel within a parallelised base-matrix column ||
| $N_{strd}$ | The number of stride units in a submatrix. Each *stride unit* $^{(※1)}$ groups a number of row chunks whose message-passing operations are processed in parallel | $N_{strd}=\lceil Z / (P_{r} \cdot W^{s}) \rceil$ |
| $N_{fg}$ | The number of messages in a row chunks, which are called stride fingers ||
| $N_{rc}$ | The number of absolute row chunks in a submatrix, i.e. the number of row chunks in a submatrix regardless of the hardware parallelism on the message-pass routing network ||
| Relative row chunk | A stride unit for a parallelised base-matrix column consists of W^{s} row chunks labeled with relative indices ranging from 0 to $W^{s}$-1. For instance, with $W^{s}$=3, a total of three elements in a stride unit is identified as relative row chunk 0, 1 and 2, respectively. ||

---
**Note:**
$(※1)$ The submatrix is evenly divided into row chunks, and the message-pass routing network services $W^s$ row chunks out of them at a time.

Global configuration for the subsequent simulation/emulation environment

In [46]:
global DEBUG_VERBOSE_LEVEL_0
global DEBUG_VERBOSE_LEVEL_1
DEBUG_VERBOSE_LEVEL_0 = False
DEBUG_VERBOSE_LEVEL_1 = False

## Row-chunk indices and word addresses generation

Let $S(i, j)$ and $S(i+1, j)$ denote the cyclic-shift amounts for a given set of submatrices. The $Z$ messages associated with a submatrix are mapped to a set of row chunks in a strided pattern. 

For a submatrix $\mathrm{B}(i, j)$, the $t$-th message represents the only nonzero element at $t$-th row of the submatrix. It is mapped to an **absolute row-chunk index** ($I^{arc}$), determined by:
$$
I^{arc}_{t} = t \bmod N_{rc}, \quad \forall t \in \{0, \dots, Z-1\}.
$$

> **Note:** Because the submatrices are circulant permutation matrices expanded from a QC-LDPC base matrix, there is exactly one nonzero element per column and per row. Therefore, the term "row index" ($I^{row}_{t}$) specifically accounts for the exact row coordinate of this single nonzero $t$-th message within the submatrix.

<br>

Since the proposed stride-scheduled message-passing approach processes $W^{s}$ consecutive row chunks in parallel (grouped together as a single stride unit), each absolute row chunk is mapped to a corresponding **relative row-chunk index** ($I^{rrc}$). This is calculated as:
$$
I^{rrc}_{t} = I^{arc}_{t} \bmod W^{s}.
$$

The index of the stride unit is calculated as:
$$
I^{strd}_{t} = \lfloor I^{arc}_{t} / W^{s} \rfloor.
$$

Furthermore, within a $P_{r}$-length row chunk, the exact left-aligned placement of the $t$-th message is defined by its **word address**:
$$
I^{word}_{t} = \left\lfloor \frac{t}{N_{rc}} \right\rfloor
$$

In [47]:
from stride_sched_utils import msgPass_env

## Configuration of the decoder architecture
- Parallelism in rows and columns of the target parity-check matrix (expanded from a given base matrix)
- To instantiate a container to emulate the message-pass buffer

In [ ]:
code = 2

if code == 1:
    deg_chk_max = 10
    deg_var_max = 3
    baseMatrix_row_num = deg_var_max
    baseMatrix_col_num = deg_chk_max

    Z = 765
    W_s = 5
    P_r = 51
    P_c = 2

    S_i_j = np.array([
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 656, 76, 132, 184, 233, 216, 490, 714, 715],
        [0, 22, 650, 359, 587, 65, 463, 635, 91, 100],
    ], dtype=np.int32)

else:
    deg_chk_max = 2
    deg_var_max = 2
    baseMatrix_row_num = deg_var_max
    baseMatrix_col_num = deg_chk_max

    Z = 16
    W_s = 4
    P_r = 2
    P_c = 1

    S_i_j = np.array([
        [0, 2],
        [7, 7]
    ], dtype=np.int32)


msgPassEnv = msgPass_env(
    msgPass_sched="stride_sched",
    baseMatrix_row_num=baseMatrix_row_num,
    baseMatrix_col_num=baseMatrix_row_num,
    Z=Z,
    W_s=W_s,
    P_c=P_c,
    P_r=P_r,
    depth_per_strd=2
)

# Message-Pass Buffer Control

## Generation of the relative row-chunk index and write address

In [49]:
import importlib
import stride_sched_utils

importlib.reload(stride_sched_utils)
from stride_sched_utils import *

Let test an example

In [50]:
initial_compMsg_vec = [t for t in range(msgPassEnv.Z)]

permV2C_base_addr_vec = [0]*2
permV2C_base_addr_vec[0] = 0 # tentative value for ease of simulation
permV2C_base_addr_vec[1] = 1 # tentative value for ease of simulation

for rel_bm_col in range(msgPassEnv.P_c):
	msgPassEnv.msgPass_buffer_ctrl.msgPass_buffer_permMsg_write(
			compMsg_vec=initial_compMsg_vec,
			rel_bm_col=rel_bm_col,
			region_addr_base=permV2C_base_addr_vec[0]   
	)
	msgPassEnv.msgPass_buffer_ctrl.msgPass_buffer_dump(rel_bm_col=rel_bm_col)


Relative Base-Matrix Column: 0
---------------------------------------------------------------------------
Abs RC   Rel RC   Stride Unit  Region   word[1:0]
---------------------------------------------------------------------------
0        0        0            0        [8, 0]
0        0        0            1        [0, 0]
4        0        1            0        [12, 4]
4        0        1            1        [0, 0]
1        1        0            0        [9, 1]
1        1        0            1        [0, 0]
5        1        1            0        [13, 5]
5        1        1            1        [0, 0]
2        2        0            0        [10, 2]
2        2        0            1        [0, 0]
6        2        1            0        [14, 6]
6        2        1            1        [0, 0]
3        3        0            0        [11, 3]
3        3        0            1        [0, 0]
7        3        1            0        [15, 7]
7        3        1            1        [0, 0]


The next section will address the cyclic shfit factor for each row chunk.

# Stride Unit Assignment

The row chunks, stride units and input sources of a page alignment unit are formulated as follows. A submatrix $b$ consists of a set of stride units, $\mathbb{S}=\{\mathbb{S}_{u} \mid u = 0, \cdots, N_{strd}-1\}$ where $N_{strd} = Z / (P_{r} \cdot W^{s})$ accounts for the number of stride units. For all $u \in \{0, \cdots, N_{strd}-1\}$, a set of consecutive row chunks is included in a stride unit, i.e. $\mathbb{S}_{u} = \{R^{u}_{0}, R^{u}_{2}, \cdots, R^{u}_{W^{s}-1}\}$ where $R^{u}_{0}$ denotes the first relative row chunk in the $u$-th stride unit, etc. Moreover, a row chunk $R^{u}_{i}$ aggregates a bunch of extrinsic messages.

Every extrinsic message is from a nonzero element in submatrix $b$, which represents one associated variable node., i.e.

$$
\forall i \in \{0, \cdots, W^{s}-1\}, \forall u \in \{0, \cdots, N_{strd}-1\}, \\
R^{u}_{i} = \{y_{o} \mid 0 \leq j \leq P_{r}-1, j \in \mathbb{Z}^{+}, o=N_{rc}*j+i+u*W^{s}\}.
$$

# Generation of Row-Chunk Cyclic-Shift Amount

This section determines the cyclic-shift amount required to permute each row chunk. Since all messages in the same row chunk are permuted by an identical shift amount, we only first define a set $E^{pg}$ containing one representative element from each absolute row chunk:
$$
E^{pg} = \{t \in [0, Z) \mid t \bmod P_r = 0\}.
$$

---

For a circulant submatrix $\mathrm{B}(i, j)$, the term $Z-S_{i, j}$ measures the cyclic distance from a given codebit index $c \in \{0, 1, \dots, Z-1\}$ to the column index of the unique nonzero element located in the $c$-th row of the submatrix. To align these messages with the proposed message-passing routing network, each bit $c$ is mapped to a relative row-chunk index $I^{rrc}_{c, i, j}$ and localised at a word address $w(c, i, j)$ within that chunk, calculated as:

$$
\begin{aligned}
I^{rrc}_{c, i, j} &= (Z - S_{i,j} + c) \bmod W^{s}, \\
w(c, i, j) &= \left\lfloor \frac{Z - S_{i,j} + c}{N_{rc}} \right\rfloor \bmod P_r, \quad \forall c \in \{0, 1, \dots, Z-1\}.
\end{aligned}
$$

---

Finally, the cyclic-shift amount for passing messages within each row chunk from the $i$-th decoding layer to the $(i+1)$-th decoding layer is calculated as:
$$
\mathrm{w\_diff}(c, i, j) =  w(c, i+1, j) - w(c, i, j), \\
\hat{S}_{c, i, t} = 
\begin{cases}
    \mathrm{w_diff}(c, i, j), & \text{if } \mathrm{w_diff}(c, i, j) \ge 0 \\
    P_{r} + \mathrm{w_diff}(c, i, j), & \text{otherwise}
\end{cases}
$$

Example of $submatrix(i=0, j=0)$ and $submatrix(i=1, j=0)$.

In [70]:
S_i = [2, 7]
Z=16
W_s=4
P_r=2
N_rc=math.ceil(Z/P_r)
arc_idx_list = np.empty(shape=(2, Z), dtype=np.uint32)
rrc_idx_list = np.empty(shape=(2, Z), dtype=np.uint32)
word_addr_list = np.empty(shape=(2, Z), dtype=np.uint32)
l1bs_shamt_list = np.empty(Z, dtype=np.uint32)
for layer_id in range(2):
    for codebit_idx in range(Z):
        S_hat = Z-S_i[layer_id]
        word_addr = math.floor(((S_hat+codebit_idx) / N_rc) % P_r)
        word_addr_list[layer_id][codebit_idx] = word_addr
        rrc_idx = (S_hat+codebit_idx) % W_s
        rrc_idx_list[layer_id][codebit_idx] = rrc_idx
        arc_idx =(S_hat+codebit_idx) % N_rc
        arc_idx_list[layer_id][codebit_idx] = arc_idx
        #print(f"codebit[{codebit_idx}\t] @ layer {layer_id} -> rel_rrc: {rrc_idx},\tword-address: {word_addr}")

for codebit_idx in range(Z):
    word_addr_diff = np.int32(word_addr_list[1][codebit_idx])-np.int32(word_addr_list[0][codebit_idx])
    l1bs_shamt =  word_addr_diff if word_addr_diff>=0 else P_r+word_addr_diff
    print(f"codebit[{codebit_idx}\t] @ layer {0} -> Abs RC: {arc_idx_list[0][codebit_idx]},\tRel RC: {rrc_idx_list[0][codebit_idx]},\tL1BS shamt: {l1bs_shamt} (diff: {word_addr_diff})")

codebit[0	] @ layer 0 -> Abs RC: 6,	Rel RC: 2,	L1BS shamt: 0 (diff: 0)
codebit[1	] @ layer 0 -> Abs RC: 7,	Rel RC: 3,	L1BS shamt: 0 (diff: 0)
codebit[2	] @ layer 0 -> Abs RC: 0,	Rel RC: 0,	L1BS shamt: 1 (diff: 1)
codebit[3	] @ layer 0 -> Abs RC: 1,	Rel RC: 1,	L1BS shamt: 1 (diff: 1)
codebit[4	] @ layer 0 -> Abs RC: 2,	Rel RC: 2,	L1BS shamt: 1 (diff: 1)
codebit[5	] @ layer 0 -> Abs RC: 3,	Rel RC: 3,	L1BS shamt: 1 (diff: 1)
codebit[6	] @ layer 0 -> Abs RC: 4,	Rel RC: 0,	L1BS shamt: 1 (diff: 1)
codebit[7	] @ layer 0 -> Abs RC: 5,	Rel RC: 1,	L1BS shamt: 0 (diff: 0)
codebit[8	] @ layer 0 -> Abs RC: 6,	Rel RC: 2,	L1BS shamt: 0 (diff: 0)
codebit[9	] @ layer 0 -> Abs RC: 7,	Rel RC: 3,	L1BS shamt: 0 (diff: 0)
codebit[10	] @ layer 0 -> Abs RC: 0,	Rel RC: 0,	L1BS shamt: 1 (diff: -1)
codebit[11	] @ layer 0 -> Abs RC: 1,	Rel RC: 1,	L1BS shamt: 1 (diff: -1)
codebit[12	] @ layer 0 -> Abs RC: 2,	Rel RC: 2,	L1BS shamt: 1 (diff: -1)
codebit[13	] @ layer 0 -> Abs RC: 3,	Rel RC: 3,	L1BS shamt: 1 (diff: -1

In [ ]:
# Cyclic-shift amount for each row chunk in a submatrix, which is used to control the L1BS components
abs_rowChunk_shamt_vec = np.zeros(
    shape=(
        baseMatrix_row_num,
        baseMatrix_col_num,
        N_rc # number of row chunks in a submatrix
    ),
    dtype=np.int32
)

page_vec = [t for t in range(N_rc)] # Generate the ascending-order labels for the row chunks in a submatrix
for bm_row in range(baseMatrix_row_num):
    for bm_col in range(baseMatrix_col_num):
        for t in page_vec:
            # To generate the cyclic shift factors for row chunks
            p0 = msgPass_buffer_ctrl_inst.page_addr_gen(t=t, l1bs_shamt=S_i_j[bm_row][bm_col])
            w0 = msgPass_buffer_ctrl_inst.word_addr_gen(t=t, l1bs_shamt=S_i_j[bm_row][bm_col])
            p1 = msgPass_buffer_ctrl_inst.page_addr_gen(t=t, l1bs_shamt=S_i_j[(bm_row+1) % baseMatrix_row_num][bm_col])
            w1 = msgPass_buffer_ctrl_inst.word_addr_gen(t=t, l1bs_shamt=S_i_j[(bm_row+1) % baseMatrix_row_num][bm_col])
            if w1 < w0:
                hat_S_i_j_r = P_r+w1-w0
            else: # w1 >= w0
                hat_S_i_j_r = w1-w0
            print(f"submatrix({bm_row}, {bm_col}), abs_row_chunk {t}: {hat_S_i_j_r}, page_addr: {p0, p1}")
            abs_rowChunk_shamt_vec[bm_row][bm_col][p0] = hat_S_i_j_r



# Algorithm - Message Passing Procedures for the Computed V2C Messages

- StrideUnit-to-message converter: To obtain the indices of stride unit and relative row chunk that $t$-th message is mapped to 
$$
\forall i \in \{0, \cdots, W^{s}-1\}, \forall u \in \{0, \cdots, N^{strd}-1\}, \\
R^{u}_{i} = \{y_{o} | 0 \leq j \leq P_{r}-1, j \in \mathbb{Z}^{+}, o=N_{rc}*j+i+u*W^{s}\}.
$$

- Message-to-StrideUnit converter, assume the cyclic shift factor is zero
  - Input: message index $t \in [0, Z)$
  - Output 1): Stride unit index, $u$
  - Output 2): Relative row chunk index, $i$
$$
\forall t \in [0, Z), \\

u = \lfloor (t \mod{(W^{s} \cdot N_{strd})}) / W^{s} \rfloor \\
i = (t \mod{(W^{s} \cdot N_{strd})}) \mod W^{s}
$$

---

Let assign sample values to all stride unit.

In [ ]:
R = [ [[0]*P_r for i in range(W_s)] for u in range(msgPass_buffer_ctrl_inst.N_strd) ]
for u in range(msgPass_buffer_ctrl_inst.N_strd):
	for i in range(W_s):
		for j in range(P_r):
			R[u][i][j] = N_rc*j+i+u*W_s
			#print(f"R[{u}][{i}][{j}] = {R[u][i][j]}")

for t in range(Z):
	u = math.floor((t % (W_s*msgPass_buffer_ctrl_inst.N_strd)) / W_s)
	i = (t % (W_s*msgPass_buffer_ctrl_inst.N_strd)) % W_s
	#print(f"t={t} u={u} i={i}")
	if t not in R[u][i]:
		print(f"Error: t={t} u={u} i={i} R[u][i]={R[u][i]}") # Error: t is not in the list of R[u][i]

# Emulation of Layer-Shift ROM

The Layer-shift ROM contains two memory regions storing:
- Cyclic-shift amount:
  - **Generation strategy:** To convert the vector of cyclic-shift amounts permuted by the indices of the absolute row chunks, into the vector of cyclic-shift amount permuted by the indices of the parallel base-matrix columns, the stride units and the relative row chunks.
- Page addresses for the cyclic shifted row chunks:
  - **Generation strategy:** To precalculate the page addresses for writing back the cyclic shifted message within $\text{submatrix}(i,j)$. There are $W^{s}$ number of memory blocks with independent controls, where the messages from every stride unit of row chunks are evenly distributed over those memory blocks. There are $N_{strd}=\lceil Z / (P_{r} \cdot W^{s}) \rceil$ stride units containing $W^{s}$ row chunks for each. Therefore, a total of $\lceil Z / P_{r} \rceil$ row chunks labelled by absolute indices in a submatrix need to have corresponding relative row-chunk indices and stride unit indices as follows.
$$
\forall r \in [0, \lceil Z / P_{r} \rceil), \\
r^{rel}(r) = r^{abs}_{r} \mod W^{s} \\
u(r) = \lfloor r^{abs}_{r} / W^{s} \rfloor
$$

In [ ]:
page_vec = [t for t in range(N_rc)] # Generate the ascending-order labels for the row chunks in a submatrix
for bm_row in range(baseMatrix_row_num):
    for bm_col in range(baseMatrix_col_num):
        # Initial step:
        abs_rowChunk_id_vec     = list()
        rel_rowChunk_id_vec     = list()
        stride_unit_id_vec      = list()
        hat_abs_rowChunk_id_vec = list()
        hat_rel_rowChunk_id_vec = list()
        hat_stride_unit_id_vec  = list()
        for t in page_vec:
            abs_rowChunk_id = msgPass_buffer_ctrl_inst.page_addr_gen(t=t, l1bs_shamt=S_i_j[bm_row][bm_col])
            rel_rowChunk_id = abs_rowChunk_id % W_s
            stride_unit_id = math.floor(abs_rowChunk_id / W_s)
            abs_rowChunk_id_vec.append(abs_rowChunk_id)
            rel_rowChunk_id_vec.append(rel_rowChunk_id)
            stride_unit_id_vec.append(stride_unit_id)
            #print(f"submatrix({bm_row}, {bm_col})_codebit: {t}|\t->\t| Row chunk for i-th layer:\t {abs_rowChunk_id} (absolute) at stride unit {stride_unit_id} converted to {rel_rowChunk_id} (relative) at stride unit {stride_unit_id}")
        
            hat_abs_rowChunk_id = msgPass_buffer_ctrl_inst.page_addr_gen(t=t, l1bs_shamt=S_i_j[(bm_row+1) % baseMatrix_row_num][bm_col])
            hat_rel_rowChunk_id = hat_abs_rowChunk_id % W_s
            hat_stride_unit_id = math.floor(hat_abs_rowChunk_id / W_s)
            hat_abs_rowChunk_id_vec.append(hat_abs_rowChunk_id)
            hat_rel_rowChunk_id_vec.append(hat_rel_rowChunk_id)
            hat_stride_unit_id_vec.append(hat_stride_unit_id)
            #print(f"submatrix({(bm_row+1)%baseMatrix_row_num}, {bm_col})_codebit: {t}|\t->\t| Row chunk for (i+1)-th layer:\t {hat_abs_rowChunk_id} (absolute) at stride unit {hat_stride_unit_id} converted to {hat_rel_rowChunk_id} (relative) at stride unit {hat_stride_unit_id}")



        # 1) absolute-to-relative row-chunk shift factor conversion
        # abs_rowChunk_shamt_vec = np.zeros(
        #     shape=(
        #         baseMatrix_row_num,
        #         baseMatrix_col_num,
        #         N_rc # number of row chunks in a submatrix
        #     ),
        #     dtype=np.int32
        # )
        #                  |
        #                  |
        #                  |
        #                  V
        # layShift_rom = np.zeros(
        # 	shape=(
        # 		self.P_c, # number of parallel base-matrix columns
        # 		self.W_s, # number of MEM blocks in a parallel base-matrix column
        # 		self.baseMatrix_row_num, # number of memory regions allocated for respective decoding layers
        # 		self.memBlk_page_depth, # number of write-back patterns for each MEM block, i.e. memory depth
        # 	),
        # 	dtype=layerSh_rom_page_dtype_0
        # )
        rel_bm_col = bm_col % P_c # baseMatrix_col_num number of base-matrix columns are evenly allocated to
                                  # the processes of those P_c number of paralle base-matrix columns.
                                  # Namely, for {baseMatrix_col_num=4, P_c=2}, the base-matrix columns' corresponding
                                  # message-passing operations are handled by the following parallel base-matrix column
                                  # deployment:
                                  #   a) Parallel (relative) base-matrix column 0: handles base-matrix columns {0, 2}
                                  #   b) Parallel (relative) base-matrix column 1: handles base-matrix columns {1, 3}
        page_addr_base = math.floor(bm_col / P_c)*msgPass_buffer_ctrl_inst.N_strd
        for abs_row_chunk in range(N_rc):
            rel_rowChunk_id = abs_row_chunk % W_s
            stride_unit = math.floor(abs_row_chunk / W_s)
            #print(f"msgPass_buffer_ctrl_inst.layShift_rom[{rel_bm_col}][{rel_rorel_rowChunk_idwChunk_id}][{bm_row}][{stride_unit}+{page_addr_base}]: {abs_rowChunk_shamt_vec[bm_row][bm_col][abs_row_chunk]}")
            msgPass_buffer_ctrl_inst.layShift_rom[rel_bm_col][rel_rowChunk_id][bm_row][stride_unit+page_addr_base] = abs_rowChunk_shamt_vec[bm_row][bm_col][abs_row_chunk]

        # 2) To generate the page addresses for writing back the cyclic shifted messages
        rel_bm_col = bm_col % P_c # baseMatrix_col_num number of base-matrix columns are evenly allocated to
                                  # the processes of those P_c number of paralle base-matrix columns.
                                  # Namely, for {baseMatrix_col_num=4, P_c=2}, the base-matrix columns' corresponding
                                  # message-passing operations are handled by the following parallel base-matrix column
                                  # deployment:
                                  #   a) Parallel (relative) base-matrix column 0: handles base-matrix columns {0, 2}
                                  #   b) Parallel (relative) base-matrix column 1: handles base-matrix columns {1, 3}
        page_addr_base = math.floor(bm_col / P_c)*msgPass_buffer_ctrl_inst.N_strd
        for abs_row_chunk in range(N_rc):
            cur_stride_unit = stride_unit_id_vec[abs_row_chunk]
            hat_stride_unit = hat_stride_unit_id_vec[abs_row_chunk]
            rel_rowChunk_id = rel_rowChunk_id_vec[abs_row_chunk]
            msgPass_buffer_ctrl_inst.page_waddr_vec[rel_bm_col][rel_rowChunk_id][bm_row][cur_stride_unit+page_addr_base]['stride_unit'] = hat_stride_unit
            msgPass_buffer_ctrl_inst.page_waddr_vec[rel_bm_col][rel_rowChunk_id][bm_row][cur_stride_unit+page_addr_base]['rel_row_chunk'] = hat_rel_rowChunk_id_vec[abs_row_chunk]
            #print(f"Submatrix({bm_row}, {bm_col}), StrideUnit_{cur_stride_unit}, RelRowChunk_{rel_rowChunk_id} -> StrideUnit_{hat_stride_unit}, RelRowChunk_{hat_rel_rowChunk_id_vec[abs_row_chunk]}: {msgPass_buffer_ctrl_inst.page_waddr_vec[rel_bm_col][rel_rowChunk_id][bm_row][cur_stride_unit+page_addr_base]}")
        
        del abs_rowChunk_id_vec
        del rel_rowChunk_id_vec
        del stride_unit_id_vec
        del hat_abs_rowChunk_id_vec
        del hat_rel_rowChunk_id_vec
        del hat_stride_unit_id_vec
        gc.collect() # For garbage collection
 


Let's confirm the content of the generated cyclic-shift amounts and page waddr values.

In [ ]:
for bm_row in range(baseMatrix_row_num):
    #print(f"-----------\nLayer {bm_row}")
    for bm_col in range(baseMatrix_col_num):
        #print(f"\t#Base-matrix column {bm_col}")
        rel_bm_col = bm_col % P_c
        page_addr_base = math.floor(bm_col / P_c)*msgPass_buffer_ctrl_inst.N_strd
        for stride_unit in range(msgPass_buffer_ctrl_inst.N_strd):
            #print(f"\t\t#Stride unit {stride_unit}")
            for rel_row_chunk in range(W_s):
                stride_unit_waddr = msgPass_buffer_ctrl_inst.page_waddr_vec[rel_bm_col][rel_row_chunk][bm_row][stride_unit+page_addr_base]['stride_unit']
                rc_waddr = msgPass_buffer_ctrl_inst.page_waddr_vec[rel_bm_col][rel_row_chunk][bm_row][stride_unit+page_addr_base]['rel_row_chunk']
                cyclic_l1bs_shamt = msgPass_buffer_ctrl_inst.layShift_rom[rel_bm_col][rel_row_chunk][bm_row][stride_unit+page_addr_base]
                #print(f"\t\t\tRelative row chunk {rel_row_chunk} "+
                #      f"page_waddr (strideUnit, rel_row_chunk): ({stride_unit_waddr}, {rc_waddr}),\tcyclic shift factor: {cyclic_l1bs_shamt}")


# Simulation of the message passing operation of the computed variable-to-check messages

---

Declaration of the message-pass buffer composed of $W^{s}$ number of memory blocks for each parallel base-matrix column.

In [ ]:
memBlk_page_depth = math.ceil(baseMatrix_col_num / P_c)*msgPass_buffer_ctrl_inst.N_strd*(baseMatrix_row_num+1) # "+1" at the last operand, "(baseMatrix_row_num+1)",
																				 # is to preserve one more memory region to store the 
                                                                                 # initial permutation for the dump verification
msgPass_buffer_wrapper = np.zeros(
	shape=(
	    P_c, # Number of parallel base-matrix columns
	    W_s, # Number of MEM blocks in a parallel base-matrix column
        baseMatrix_row_num+1, # Number of memory regions allocated for respective decoding layers, with an additional layer preserved for verification
	    memBlk_page_depth, # Number of write-back patterns for each MEM block, i.e. memory depth
        P_r # Number of messages packed in one memory page
	),
	dtype=np.int32
)

To preload the sample channel messages (input LLR values) to the message-pass buffers where the message positions are assumed to be have all aligned with the cyclic-shift consequences of the submatrices in the first decoding layer.

The input LLR loader is implemented by the following steps:
1. To generate the $P_{c}$ sets of input LLR messages, each of which contains $Z$ messages assigned with the values: $\{0, 1, \cdots, Z-1\}$.
2. To rearrange the those two sets' permutations according to the cyclic shift factors of their respective submatrices.
3. To load the cyclic shifted LLR messages to the corresponding memory blocks.

In [ ]:
for bm_row in range(baseMatrix_row_num+1):
    for bm_col in range(baseMatrix_col_num):
        #in_llr_vec = np.arange(0, Z, dtype=np.int32)
        #in_llr_vec = np.roll(in_llr_vec, S_i_j[bm_row][bm_col])
        #print(f"submatrix({bm_row}, {bm_col}) in_llr_vec={in_llr_vec}")

        rel_bm_col = bm_col % P_c
        rel_bm_col_cnt = math.floor(bm_col / P_c)
        page_addr_base = rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd
        for t in range(Z):
            u = math.floor((t % (W_s*msgPass_buffer_ctrl_inst.N_strd)) / W_s)
            rel_row_chunk = (t % (W_s*msgPass_buffer_ctrl_inst.N_strd)) % W_s
            word_addr = msgPass_buffer_ctrl_inst.word_addr_gen(t=t, l1bs_shamt=0)
            msgPass_buffer_wrapper[rel_bm_col][rel_row_chunk][bm_row][u+page_addr_base][word_addr] = t
            #print(f"msgPass_buffer_wrapper[rel_bm_col:{rel_bm_col}][rel_row_chunk:{rel_row_chunk}][bm_row:{bm_row}][page_addr:strideUnit_{u}+base_addr_{page_addr_base}][word_addr:{word_addr}]={t}")


---

**Ongoing:** The MUX4 should be inserted before this line:

```python
y_fetch_compV2C = msgPass_buffer_wrapper[rel_bm_col][rel_row_chunk][layer_next][stride_unit+rel_bm_col_cnt][j]
```

In [ ]:
#compV2C_base_addr = 0 # tentative value for ease of simulation
#permV2C_base_addr = N_rc+1 # tentative value for ease of simulation
#page_raddr = 0 + compV2C_base_addr
prefix_for_verification = 10**(math.ceil(math.log10(Z-1)))

def msgPass_stride_sched(
	layer: int,
	layer_next: int,
	rel_bm_col: int, # Relative base-matrix column in {0, ..., P_{c}-1}
	rel_bm_col_cnt: int # To count the number of base-matrix columns which have been processed by the underlying message-pass routing network
						# Note that P_{c} number of independent message-pass routing networks deployed on the decoder subsystem
):
	page_raddr = 0
	page_addr_base = rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd # Each base-matrix column requires U pages of row chunk messages on every memory block
	for stride_unit in range(msgPass_buffer_ctrl_inst.N_strd):
		for rel_row_chunk in range(W_s):
			S_bs = msgPass_buffer_ctrl_inst.layShift_rom[rel_bm_col][rel_row_chunk][layer][stride_unit+page_addr_base]
			#print(f"\t\t\trel_bm_col_cnt: {rel_bm_col_cnt}, strideUnit_{stride_unit}, rel_row_chunk_{rel_row_chunk}, S_bs: {S_bs}")
			stride_unit_waddr = msgPass_buffer_ctrl_inst.page_waddr_vec[rel_bm_col][rel_row_chunk][layer][stride_unit+page_addr_base]['stride_unit']
			rc_waddr = msgPass_buffer_ctrl_inst.page_waddr_vec[rel_bm_col][rel_row_chunk][layer][stride_unit+page_addr_base]['rel_row_chunk']
			for j in range(P_r):
				# Fetch the compV2C
				y_fetch_compV2C = msgPass_buffer_wrapper[rel_bm_col][rel_row_chunk][layer][stride_unit+rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd][j]

				# Column-wise cyclic shift outputing the permV2C
				hat_j = int((j + S_bs) % P_r)

				# Wirte back the permV2C
				msgPass_buffer_wrapper[rel_bm_col][rc_waddr][layer_next][stride_unit_waddr+rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd][hat_j] = y_fetch_compV2C+prefix_for_verification*layer
			
			# == For fine-grained debug (1) =========================================================================================
			if(DEBUG_VERBOSE_LEVEL_1==True):
				print(f"\t\t\t\tFetch compV2Cs from msgPass_buffer_wrapper[{rel_bm_col}][{rel_row_chunk}][{layer}][{stride_unit+rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd}]")
				temp_pre = msgPass_buffer_wrapper[rel_bm_col][rel_row_chunk][layer][stride_unit+rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd]
				temp_post = msgPass_buffer_wrapper[rel_bm_col][rc_waddr][layer_next][stride_unit_waddr+rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd]
				print(f"\t\t\t\tS_bs: {S_bs}, compV2C_vec: {temp_pre} -> write permV2C_vec: {temp_post} back to memblk[{rc_waddr}], layer-{layer_next}, rel_bm_col-{rel_bm_col}, strideUnit-{stride_unit_waddr} w/ base_addr-{rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd}")
			# == End of fine-grained debug (1) ======================================================================================
			page_raddr += 1

def msgPass_buffer_dump(layer:int, rel_bm_col: int, rel_bm_col_cnt: int):
	print(f"\tLayer-{layer}, Relative Base-Matrix Column_{rel_bm_col} w/ base region_{rel_bm_col_cnt}")
	page_addr_base = rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd
	for stride_unit in range(msgPass_buffer_ctrl_inst.N_strd):
		print(f"\t\tStride unit {stride_unit}")
		for memblk in range(W_s):
			print(f"\t\t\tmemblk_{memblk} -> {msgPass_buffer_wrapper[rel_bm_col][memblk][layer][stride_unit+page_addr_base]}")

def msgPass_buffer_reset_with_x(layer:int, rel_bm_col: int, rel_bm_col_cnt: int):
	# Let define value Z as the X value (considered as uncertainty or noise)
	page_addr_base = rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd
	for stride_unit in range(msgPass_buffer_ctrl_inst.N_strd):
		for memblk in range(W_s):
			msgPass_buffer_wrapper[rel_bm_col][memblk][layer][stride_unit+page_addr_base]=Z

if(DEBUG_VERBOSE_LEVEL_0==True):
	print("\n------------------------------------------------------------------------")
	print(f"msgPass_buffer before cyclic shift")
	for layer in range(baseMatrix_row_num):
		for rel_bm_col_cnt in range(math.ceil(baseMatrix_col_num / P_c)):
			for rel_bm_col in range(P_c):
				msgPass_buffer_dump(layer=layer, rel_bm_col=rel_bm_col, rel_bm_col_cnt=rel_bm_col_cnt)
	print("------------------------------------------------------------------------")
	
for layer in range(baseMatrix_row_num):
	layer_next = (layer+1) % baseMatrix_row_num
	for bm_col in range(baseMatrix_col_num):
		rel_bm_col = bm_col % P_c
		rel_bm_col_cnt = math.floor(bm_col / P_c)
		msgPass_stride_sched(layer=layer, layer_next=layer_next, rel_bm_col=rel_bm_col, rel_bm_col_cnt=rel_bm_col_cnt)

if(DEBUG_VERBOSE_LEVEL_0==True):
	print("\n------------------------------------------------------------------------")
	print(f"msgPass_buffer after cyclic shift:")
	for layer in range(baseMatrix_row_num):
		layer_next = (layer+1) % baseMatrix_row_num
		for rel_bm_col_cnt in range(math.ceil(baseMatrix_col_num / P_c)):
			for rel_bm_col in range(P_c):
				msgPass_buffer_dump(layer=layer_next, rel_bm_col=rel_bm_col, rel_bm_col_cnt=rel_bm_col_cnt)


# Verification of the Simulation Results

In [ ]:
def msgPass_buffer_dump_verification(layer_num: int, first_layer: int, last_layer: int, rel_bm_col: int, rel_bm_col_cnt: int) -> bool:
	is_verification_passed = True
	prefix_for_lastLayer_verification = prefix_for_verification*layer_num
	page_addr_base = rel_bm_col_cnt*msgPass_buffer_ctrl_inst.N_strd
	for stride_unit in range(msgPass_buffer_ctrl_inst.N_strd):
		for memblk in range(W_s):
			if((msgPass_buffer_wrapper[rel_bm_col][memblk][first_layer][stride_unit+page_addr_base]<prefix_for_lastLayer_verification).any()):
				print(f"(Loss of message passing) Layer-{layer}, Relative Base-Matrix Column_{rel_bm_col} w/ base region_{rel_bm_col_cnt}, Stride unit {stride_unit}, memblk_{memblk} -> {msgPass_buffer_wrapper[rel_bm_col][memblk][first_layer][stride_unit+page_addr_base]}")
				is_verification_passed=False

			if(
				np.not_equal(
					msgPass_buffer_wrapper[rel_bm_col][memblk][first_layer][stride_unit+page_addr_base]-prefix_for_lastLayer_verification,
					msgPass_buffer_wrapper[rel_bm_col][memblk][last_layer+1][stride_unit+page_addr_base]
				).any()
			):
				print(f"(Inconsistency) Layer-{layer}, Relative Base-Matrix Column_{rel_bm_col} w/ base region_{rel_bm_col_cnt}, Stride unit {stride_unit}, memblk_{memblk} -> {msgPass_buffer_wrapper[rel_bm_col][memblk][first_layer][stride_unit+page_addr_base]-prefix_for_lastLayer_verification} != {msgPass_buffer_wrapper[rel_bm_col][memblk][last_layer+1][stride_unit+page_addr_base]}")
				is_verification_passed=False

	return is_verification_passed

layer_num = baseMatrix_row_num
first_layer = 0
last_layer = layer_num-1			
for rel_bm_col_cnt in range(math.ceil(baseMatrix_col_num / P_c)):
	for rel_bm_col in range(P_c):
		is_verification_passed = msgPass_buffer_dump_verification(
			layer_num=layer_num,
			first_layer=first_layer,
			last_layer=last_layer,
			rel_bm_col=rel_bm_col,
			rel_bm_col_cnt=rel_bm_col_cnt
		)

		if(is_verification_passed==True):
			print(f"Relative base-matrix column {rel_bm_col} with rel_bm_col_cnt-{rel_bm_col_cnt}: Passed")
		else:
			print(f"Relative base-matrix column {rel_bm_col} with rel_bm_col_cnt-{rel_bm_col_cnt}: Failed")

# List of all L1BS cyclic-shift amounts

In [ ]:
for bm_row in range(baseMatrix_row_num):
    print(f"Layer: {bm_row}")
    for rel_bm_col in range(P_c):
        print(f"rel_bm_col: {rel_bm_col}")
        for rel_rowChunk_id in range(W_s):
            print(f"Relative row chunk ({rel_rowChunk_id}):")
            for idx, shamt in enumerate(msgPass_buffer_ctrl_inst.layShift_rom[rel_bm_col][rel_rowChunk_id][bm_row]):
                abs_bm_col_id = math.floor(idx / msgPass_buffer_ctrl_inst.N_strd)
                stride_unit_id = idx % msgPass_buffer_ctrl_inst.N_strd
                print(f"Absolute BM-Col ({abs_bm_col_id}), Stride unit ({stride_unit_id}): {shamt}")
				